# R117 design audit companion

This notebook reproduces the three read-only diagnostic scripts and loads
their evidence. It does not start training, modify experiment code, or save
weights. Detailed interpretation is in `FINDINGS.md`.

The key findings are a mislabeled R109-to-R117 preview cascade, a normal/SDF
polarity mismatch, inactive geometry-consistency training, and a whole-object
normal-distance average that is insensitive to actual x/y translation.


## Context and assumptions

The 72-scene diagnostic balances 12 shape families and two polarities. It is
drawn from the existing development validation pool, not a new sealed holdout.
Oracle geometry tests isolate coordinate errors and are not deployment results.
The real image reference is the pre-R109 source verified by reproducing R109.
The center-nine radial profile measures edge oscillation, not circularity truth.


In [1]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
here = Path(r'D:\GMM_2_Final\Experiments\R117_CenterLockedSubpixelScratch\analysis\design_audit_20260905')


## Reproduce the diagnostics

In [2]:
for script in ['diagnose.py', 'preview_source_audit.py', 'render_mechanism_audit.py']:
    completed = subprocess.run([sys.executable, str(here / script)], capture_output=True, text=True, encoding='utf-8', errors='replace')
    if completed.returncode:
        raise RuntimeError(completed.stderr[-6000:] + completed.stdout[-3000:])
    print(script, 'passed')

diagnose.py passed


preview_source_audit.py passed


render_mechanism_audit.py passed


## Check protected inputs and actual supervision

In [3]:
audit = json.loads((here / 'audit_results.json').read_text())
assert audit['protected_files_unchanged']
print('Training source, final checkpoint and data manifest unchanged:', audit['protected_files_unchanged'])
print('Detector E6/E42 identical:', audit['checkpoint_freezing']['all_detector_tensors_bit_identical_E6_E42'])
pd.DataFrame([{ 'active': g['active_group'], 'position_pair_value':g['terms']['pair_sdf']['value'], 'position_pair_gradient':g['terms']['pair_sdf']['active_group_gradient_norm']} for g in audit['gradients']['pair_terms']])

Training source, final checkpoint and data manifest unchanged: True
Detector E6/E42 identical: True


,active,position_pair_value,position_pair_gradient
0,width,2.065212,0.0
1,shift,2.065212,0.0
2,detail,2.065212,0.0


## Check normal polarity and target reachability

In [4]:
summaries = audit['model_data']['summary']
pd.DataFrame([{ 'polarity': name, 'wrong_normal_fraction': summaries[name]['normal_wrong_sign_fraction']['mean'], 'native_oracle_sampling_error_px': summaries[name]['true_sdf_native_normal_sampling_error_px']['mean'], 'sign_corrected_oracle_sampling_error_px': summaries[name]['true_sdf_sign_corrected_sampling_error_px']['mean'], 'unreachable_width_target_fraction': summaries[name]['target_gain_beyond_actuator_fraction']['mean']} for name in ['bright','dark']])

,polarity,wrong_normal_fraction,native_oracle_sampling_error_px,sign_corrected_oracle_sampling_error_px,unreachable_width_target_fraction
0,bright,0.016614,0.141481,0.124473,0.650394
1,dark,0.977566,4.079768,0.191900,0.679914


## Reconcile the real preview inputs

In [5]:
preview = json.loads((here / 'preview_source_results.json').read_text())
assert all(r['r109_reproduced_vs_configured_max_gray'] == 0 for r in preview['images'])
pd.DataFrame(preview['images'])[['image','raw_vs_configured_mean_gray','r109_reproduced_vs_configured_max_gray','r117_direct_change_mean_gray','r117_cascade_vs_direct_mean_gray']]

,image,raw_vs_configured_mean_gray,r109_reproduced_vs_configured_max_gray,r117_direct_change_mean_gray,r117_cascade_vs_direct_mean_gray
0,5x_dense_raw.bmp,0.002117,0.0,1.020489,0.002663
1,20x_mixed_raw.bmp,3.405262,0.0,1.128254,3.496357
2,50x_array_raw.bmp,4.646877,0.0,1.493301,5.073872
3,pillar_center9_50x_raw.bmp,2.620783,0.0,0.561404,2.745114


## Check the claimed center metric and ring mechanism

In [6]:
display(pd.DataFrame(audit['analytic']['centroid_metric_blindness']))
render = json.loads((here / 'render_mechanism_results.json').read_text())
pd.Series({name:p['radial_reversal_gray'] for name,p in render['pillar_profiles'].items()},name='Radial reversal, gray')

,true_center_shift_px,dx,dy,reported_center_lock_error
0,0.000000,0.0,0,0.000000
1,0.500000,0.5,0,0.000160
2,1.000000,1.0,0,0.001380
3,2.000000,2.0,0,0.006957
4,1.000000,0.0,1,0.001380
5,1.414214,1.0,1,0.002812


Raw                       0.000000
R117                      0.393609
geometry_anchor           0.014276
width_proposal            0.076824
continuous_width          0.399503
no_projection             0.003785
normal_sign_consistent    0.181733
Name: Radial reversal, gray, dtype: float64

## Interpretation

Training completion and finite tensors did not establish the scientific
properties intended by the design. The normal sign, actual translation
objective, gradient path and output continuity need explicit mechanistic
checks. The frozen projection intervention reduces a ring when bypassed on
one crop; it does not select a deployable alternative. A lower horizontal
gradient energy alone is not evidence of poorer detail fidelity. The broader
model comparison still needs same-input ACM and localization validation.
